# 13. Spectrum Occupancy Prediction based on Adaptive Recurrent Neural Networks

Implements **DC prediction with two RNNs and state identification** from the paper:
**"Spectrum Occupancy Prediction based on adaptive Recurrent Neural Networks"** (IEEE WCNC 2023).

## Paper idea (adapted to our setting)
- **Problem:** A single RNN trained on biased data (e.g. mostly low occupancy) tends to under-predict when occupancy is high.
- **Solution:** Use **two RNNs** — one for **high** occupancy (state S1), one for **low** (state S0). **State identification** chooses which RNN to use based on recent observed values.
- **State identification:** If the mean of the last `w` hours of input ≥ threshold η → state = high (use RNN_high), else state = low (use RNN_low). Paper uses η=0.8, w=1 for duty cycle 0–1; we use η=0.5 (50% AU) and w=6 hours in 0–1 scaled space.

## Same setup as notebooks 10–12
- **Data:** `work_dir/final/training/` and `work_dir/final/testing/` (6 bands, same as 10).
- **Input:** last **72 hours** (3 days); **Output:** next **24 hours** (next day).
- **Metrics:** MAE, RMSE, MASE.
- **Models:** Adaptive Two-RNN (proposed), Conventional single RNN, Naive baseline.
- **Visuals:** Results table, bar charts, improvement over naive, predicted vs actual (dashed), mean profile, per-hour MAE, residuals, key insights, one-line summary.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'CPU cores: {os.cpu_count()}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e:
        print('GPU config:', e)
else:
    print('No GPU. On Apple Silicon: pip install tensorflow-metal')

USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
if not final_dir.exists():
    raise FileNotFoundError("Final dir not found: " + str(final_dir))
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("Training or testing directory not found")

class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
print(f"Bands: {class_options}")

LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Lookback={LOOKBACK}h, Forecast={FORECAST_HORIZON}h")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")
            continue
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df: pd.DataFrame, test_df: pd.DataFrame, lookback: int = 72, forecast_horizon: int = 24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    thresholds = sorted(train_df["threshold_dbm"].unique())
    if len(thresholds) > 1:
        train_df = train_df[train_df["threshold_dbm"] == thresholds[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == thresholds[0]].copy()
    X_train_list, y_train_list = [], []
    X_test_list, y_test_list = [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_train_list.append(tr[i:i+lookback])
            y_train_list.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                input_slice = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                hours_from_train = max(0, lookback - d * forecast_horizon)
                if hours_from_train > 0:
                    input_slice = np.concatenate([tr[-hours_from_train:], te[0:d*forecast_horizon]])
                else:
                    input_slice = te[d*forecast_horizon - lookback:d*forecast_horizon]
            target_slice = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(input_slice) == lookback and len(target_slice) == forecast_horizon:
                X_test_list.append(input_slice)
                y_test_list.append(target_slice)
    if not X_train_list or not X_test_list:
        return np.array([]), np.array([]), np.array([]), np.array([])
    X_train = np.array(X_train_list).reshape(-1, lookback, 1)
    y_train = np.array(y_train_list)
    X_test = np.array(X_test_list).reshape(-1, lookback, 1)
    y_test = np.array(y_test_list)
    return X_train, y_train, X_test, y_test

In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    train_df = load_data_for_band(band, "training")
    test_df = load_data_for_band(band, "testing")
    if not train_df.empty and not test_df.empty:
        train_data_by_band[band] = train_df
        test_data_by_band[band] = test_df
        print(f"Band {band}: Train={len(train_df)} rows, Test={len(test_df)} rows")
print(f"\nLoaded {len(train_data_by_band)} bands")

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
for band in class_options:
    if band not in train_data_by_band:
        continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(
        train_data_by_band[band], test_data_by_band[band], lookback=LOOKBACK, forecast_horizon=FORECAST_HORIZON
    )
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
        print(f"Band {band}: Train seq={len(X_tr)}, Test seq={len(X_te)}")

X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"\nTotal: Train {len(X_train)}, Test {len(X_test)}")
print(f"Shapes: X_train {X_train.shape}, y_train {y_train.shape}")

In [ ]:
# Normalize (0-1) for RNN training; state identification uses same scale
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_flat = X_train.reshape(-1, 1)
X_test_flat = X_test.reshape(-1, 1)
y_train_flat = y_train.reshape(-1, 1)
y_test_flat = y_test.reshape(-1, 1)
scaler_X.fit(X_train_flat)
scaler_y.fit(y_train_flat)
X_train_scaled = scaler_X.transform(X_train_flat).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test_flat).reshape(X_test.shape)
y_train_scaled = scaler_y.transform(y_train_flat).reshape(y_train.shape)
y_test_scaled = scaler_y.transform(y_test_flat).reshape(y_test.shape)
print("Data normalized (0-1).")

## State identification and two-RNN training (paper: high DC vs low DC)

**Paper (WCNC 2023):** State S0 = low DC, S1 = high DC. State identification uses past w O-DCs; if they are more than threshold η → S1, else S0. Two RNNs (one per state); each RNN: input layer (sl neurons), hidden layer (n=10, tanh), output (sigmoid). Paper uses sl=2, η=0.8, w=1 for one-step DC prediction.

**This notebook (adapted):**
- **η (eta):** threshold in 0–1 scale; state = high if mean of last `w` hours ≥ η. Paper η=0.8; we use 0.5 (50% AU).
- **w:** number of recent hours for state. Paper w=1; we use w=6 for stability.
- **RNN:** Same LSTM(10)→Dense(24) as notebook 10 (multi-step 24h output). Input = 72h (lookback).
- Split training into high-state and low-state; train RNN_high and RNN_low.

In [ ]:
ETA = 0.5   # threshold for "high" state (0-1 scale). Paper uses 0.8 for DC.
W_STATE = 6  # number of recent hours for state identification. Paper w=1.

def get_state_high(X_scaled: np.ndarray, eta: float = ETA, w: int = W_STATE) -> np.ndarray:
    """True = high state (use RNN_high), False = low state (use RNN_low). X_scaled: (n, lookback, 1)."""
    # mean of last w hours, shape (n,)
    last_w = X_scaled[:, -w:, 0]
    mean_last_w = np.mean(last_w, axis=1)
    return mean_last_w >= eta

# Split training into high and low state
state_high_train = get_state_high(X_train_scaled, ETA, W_STATE)
X_train_high = X_train_scaled[state_high_train]
y_train_high = y_train_scaled[state_high_train]
X_train_low = X_train_scaled[~state_high_train]
y_train_low = y_train_scaled[~state_high_train]
print(f"Training split: high-state samples = {len(X_train_high)}, low-state samples = {len(X_train_low)}")
if len(X_train_high) < 2 or len(X_train_low) < 2:
    print("WARNING: One state has very few samples; training may be poor.")

In [ ]:
def create_vanilla_lstm(lookback: int, forecast_horizon: int):
    """Single RNN: LSTM(10) -> Dense(forecast_horizon). Same as paper Fig 3 / notebook 10."""
    model = keras.Sequential([
        layers.LSTM(10, activation='tanh', input_shape=(lookback, 1)),
        layers.Dense(forecast_horizon, activation='linear')
    ])
    return model

def naive_predictor(X: np.ndarray, forecast_horizon: int):
    last_values = X[:, -1, 0]
    return np.tile(last_values.reshape(-1, 1), (1, forecast_horizon))

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

In [ ]:
EPOCHS = 60
BATCH_SIZE = 128 if USE_GPU else 32
LR = 0.05

# 1) Train RNN for high state
model_high = create_vanilla_lstm(LOOKBACK, FORECAST_HORIZON)
model_high.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss='mse', metrics=['mae'])
if len(X_train_high) >= 2:
    model_high.fit(X_train_high, y_train_high, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)
else:
    model_high.fit(X_train_scaled, y_train_scaled, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)

# 2) Train RNN for low state
model_low = create_vanilla_lstm(LOOKBACK, FORECAST_HORIZON)
model_low.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss='mse', metrics=['mae'])
if len(X_train_low) >= 2:
    model_low.fit(X_train_low, y_train_low, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)
else:
    model_low.fit(X_train_scaled, y_train_scaled, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)
print("Two RNNs (high / low) trained.")

In [ ]:
# 3) Conventional single RNN (all training data)
model_conventional = create_vanilla_lstm(LOOKBACK, FORECAST_HORIZON)
model_conventional.compile(optimizer=keras.optimizers.Adam(learning_rate=LR), loss='mse', metrics=['mae'])
model_conventional.fit(X_train_scaled, y_train_scaled, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)
print("Conventional (single) RNN trained.")

In [ ]:
# Adaptive prediction: for each test sample, choose RNN_high or RNN_low by state
state_high_test = get_state_high(X_test_scaled, ETA, W_STATE)
y_pred_adaptive_scaled = np.zeros_like(y_test_scaled)
for i in range(len(X_test_scaled)):
    x = X_test_scaled[i:i+1]
    if state_high_test[i]:
        y_pred_adaptive_scaled[i] = model_high.predict(x, verbose=0)[0]
    else:
        y_pred_adaptive_scaled[i] = model_low.predict(x, verbose=0)[0]
y_pred_adaptive = scaler_y.inverse_transform(y_pred_adaptive_scaled.reshape(-1, 1)).reshape(y_test.shape)

# Conventional and naive
y_pred_conventional_scaled = model_conventional.predict(X_test_scaled, verbose=0)
y_pred_conventional = scaler_y.inverse_transform(y_pred_conventional_scaled.reshape(-1, 1)).reshape(y_test.shape)
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)  # raw scale, same as y_test

print("Predictions: Adaptive Two-RNN, Conventional RNN, Naive.")

In [ ]:
# Metrics and results table (same format as notebook 10)
mae_adapt = calculate_mae(y_test, y_pred_adaptive)
rmse_adapt = calculate_rmse(y_test, y_pred_adaptive)
mase_adapt = calculate_mase(y_test, y_pred_adaptive, y_train)
mae_conv = calculate_mae(y_test, y_pred_conventional)
rmse_conv = calculate_rmse(y_test, y_pred_conventional)
mase_conv = calculate_mase(y_test, y_pred_conventional, y_train)
mae_naive = calculate_mae(y_test, y_pred_naive)
rmse_naive = calculate_rmse(y_test, y_pred_naive)
mase_naive = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "Adaptive Two-RNN", "MAE": mae_adapt, "RMSE": rmse_adapt, "MASE": mase_adapt},
    {"Model": "Conventional RNN", "MAE": mae_conv, "RMSE": rmse_conv, "MASE": mase_conv},
    {"Model": "Naive Baseline", "MAE": mae_naive, "RMSE": rmse_naive, "MASE": mase_naive},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (same format as notebook 10)")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(f"\n{results_df.to_string(index=False)}")
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
# 1. Bar charts: MAE, RMSE, MASE by model
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: metric comparison', y=1.02, fontsize=12)
plt.show()

In [ ]:
# 2. Improvement over Naive Baseline (%)
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Predicted vs Actual: up to 3 test samples (actual as dashed)
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_adaptive[i], '-', linewidth=1.6, label='Adaptive Two-RNN')
    ax.plot(hours, y_pred_conventional[i], '-', linewidth=1.2, label='Conventional RNN', alpha=0.8)
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual (dashed)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

# Mean 24h profile
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_adaptive.mean(axis=0), '-', linewidth=1.6, label='Adaptive Two-RNN (mean)')
ax.plot(hours, y_pred_conventional.mean(axis=0), '-', linewidth=1.2, label='Conventional RNN (mean)', alpha=0.8)
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4. Per-hour MAE
mae_per_hour_adapt = np.abs(y_test - y_pred_adaptive).mean(axis=0)
mae_per_hour_conv = np.abs(y_test - y_pred_conventional).mean(axis=0)
mae_per_hour_naive = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour_adapt, '-o', label='Adaptive Two-RNN', markersize=4)
ax.plot(hours, mae_per_hour_conv, '-o', label='Conventional RNN', markersize=4, alpha=0.8)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. Residuals: Adaptive Two-RNN vs Naive
residuals_best = (y_test - y_pred_adaptive).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: Adaptive Two-RNN')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): Adaptive = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         Adaptive = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **Adaptive Two-RNN (paper):** State identification (η, w) selects high- or low-occupancy RNN to reduce biased error when data is skewed (e.g. mostly low AU).
- **Improvement over naive:** Positive % means the model beats the last-value baseline.
- **Mean profile:** Alignment of predicted and actual mean profiles indicates good capture of daily pattern.
- **Per-hour MAE:** Identifies hours that are harder to predict.
- **Residuals:** Centered, symmetric distributions indicate unbiased predictions.

In [ ]:
# One-line summary (same format as notebook 10)
best_row = results_df[results_df['Model'] == 'Adaptive Two-RNN'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"Best model: Adaptive Two-RNN (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}).")
print(f"Improvement over Naive: MAE {imp_mae:+.1f}%.")